In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import (
    col,
    explode,
    to_timestamp,
    from_unixtime,
    to_date,
)


BRONZE_QUOTES_PATH = (
    "s3a://kafka-spark-stock-project-kris/"
    "stock-market/bronze/rest/quote_snapshots/"
)


def main():

    spark = (
        SparkSession.builder
        .appName("QuoteSnapshotsSilverPreview")
        .config("spark.sql.session.timeZone", "UTC")
        .config(
            "spark.hadoop.fs.s3a.aws.credentials.provider",
            "org.apache.hadoop.fs.s3a.auth.IAMInstanceCredentialsProvider",
        )
        .getOrCreate()
    )

    spark.sparkContext.setLogLevel("WARN")


    # -----------------------------------------------------
    # Read Bronze JSON batches
    # -----------------------------------------------------

    bronze_df = (
        spark.read
        .option("recursiveFileLookup", "true")
        .json(BRONZE_QUOTES_PATH)
    )

    print("\nBronze batch count:")
    print(bronze_df.count())

    print("\nBronze schema:")
    bronze_df.printSchema()


    # -----------------------------------------------------
    # Each Bronze file contains:
    #
    # {
    #   batch_id: ...,
    #   collected_at_utc: ...,
    #   records: [
    #       {AAPL quote},
    #       {MSFT quote},
    #       ...
    #   ]
    # }
    #
    # explode() converts the records array into one
    # row per stock snapshot.
    # -----------------------------------------------------

    exploded_df = (
        bronze_df
        .withColumn(
            "quote",
            explode(col("records"))
        )
    )


    # -----------------------------------------------------
    # Build clean Silver representation
    # -----------------------------------------------------

    silver_df = (
        exploded_df

        .select(
            col("batch_id"),
            col("schema_version"),
            col("source"),

            to_timestamp(
                col("collected_at_utc")
            ).alias("collected_at"),

            col("quote.symbol").alias("symbol"),

            col("quote.current_price")
                .cast("double")
                .alias("current_price"),

            col("quote.change")
                .cast("double")
                .alias("change"),

            col("quote.percent_change")
                .cast("double")
                .alias("percent_change"),

            col("quote.day_open")
                .cast("double")
                .alias("day_open"),

            col("quote.day_high")
                .cast("double")
                .alias("day_high"),

            col("quote.day_low")
                .cast("double")
                .alias("day_low"),

            col("quote.previous_close")
                .cast("double")
                .alias("previous_close"),

            col("quote.quote_timestamp")
                .cast("long")
                .alias("quote_timestamp_unix"),

            col("quote.raw_quote")
                .alias("raw_quote"),
        )

        # Finnhub quote timestamp is Unix seconds
        .withColumn(
            "quote_timestamp",
            to_timestamp(
                from_unixtime(
                    col("quote_timestamp_unix")
                )
            )
        )

        .withColumn(
            "snapshot_date",
            to_date(col("collected_at"))
        )
    )


    print("\nSilver quote record count:")
    print(silver_df.count())

    print("\nSilver quote schema:")
    silver_df.printSchema()

    print("\nSample quote snapshots:")

    (
        silver_df
        .orderBy(
            col("collected_at").desc(),
            col("symbol")
        )
        .select(
            "symbol",
            "collected_at",
            "current_price",
            "change",
            "percent_change",
            "day_open",
            "day_high",
            "day_low",
            "previous_close",
            "quote_timestamp",
        )
        .show(
            30,
            truncate=False
        )
    )

    spark.stop()


if __name__ == "__main__":
    main()